# Sieć neuronowa — PyTorch

Od tego momentu wkraczamy w bardziej złożone modele i ich implementacja od podstaw wymaga znajomości wielu detali technicznych (np. cachowanie wartości funkcji aktywacji nodów podczas forward propagation), jeśli ktoś jest ambitny polecam zrobić cały program w Pythonie do uczenia sieci neuronowej (pokrótce: inicjalizacja parametrów, forward pass, backward pass) jako projekt z tematyki ML.
<br> <br>
Ponieważ nie zdążymy przejść przez szczegóły, od tego momentu będziemy korzystać z tych modeli jak z "black-box" - będzie nas interesowała ich poprawna architektura, wymiar wejść, wymiar wyjść, dane treningowe. Te bardzo szczegółowe operacje jak iteracyjne uczenie oddamy w ręce popularnych bibliotek do deep learningu — w tym notebooku będziemy korzystać z **PyTorch**. Oczywiście absolutny brak zrozumienia co się dzieje pod podszewką takich modeli jest niewskazany i dlatego postaramy się zrozumieć chociaż pewne koncepty, intuicje związane z tymi modelami - będą one pokazywane na slajdach.
<br><br>
W tym zadaniu nadal będziemy zajmować się naszym problemem rozpoznawania czy na zdjęciu jest kot czy go nie ma - podobnie jak w zadaniu regresji logistycznej.

## Ściągnięcie bibliotek i danych

**Proszę odkomentować i wykonać poniższą komórkę jeśli używają państwo Colaba lub MyBinder!**

In [ ]:
# import os
# if not os.path.exists('utils.py'):
#     !curl -O https://raw.githubusercontent.com/NXTRSS/MachineLearningCourse/main/utils.py
# from utils import check_and_download_data
# check_and_download_data(files_to_check=["catvsnotcat"])

In [ ]:
# === Konfiguracja środowiska ===
# Pakiety są już zainstalowane w środowisku nn_pytorch.
# Na Colabie — odkomentuj poniższe linie jeśli czegoś brakuje:
# !pip install -q torch torchinfo scikit-learn pillow ipywidgets matplotlib

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from PIL import Image
import random
import urllib
import os
import io
import pickle
from datetime import datetime
from torchinfo import summary as torchinfo_summary

# --- bcolors i SelectFilesButton (wbudowane, bez zależności od TF) ---
from ipywidgets import widgets

class bcolors:
    HEADER = '\033[95m'; OKBLUE = '\033[94m'; OKCYAN = '\033[96m'
    OKGREEN = '\033[92m'; WARNING = '\033[93m'; FAIL = '\033[91m'
    ENDC = '\033[0m'; BOLD = '\033[1m'; UNDERLINE = '\033[4m'

class SelectFilesButton(widgets.FileUpload):
    def __init__(self):
        super().__init__(accept='image/*', multiple=True, description='Wybierz zdjęcia',
                         button_style='warning', layout=widgets.Layout(width='auto'))

# ---------- Wybór urządzenia (MPS na Apple Silicon / CUDA na GPU / CPU) ----------
device = torch.device(
    'mps' if torch.backends.mps.is_available()
    else 'cuda' if torch.cuda.is_available()
    else 'cpu'
)
print(f"Urządzenie: {device}")

#### Ściągnięcie danych
**Proszę pamiętać, że należy ściągnąć plik catvsnotcat.pkl przed zajęciami!**

In [ ]:
file_path = 'catvsnotcat.pkl'

with open(file_path, 'rb') as file:
    all_data = pickle.load(file)

#### Zrzutowanie wszystkich zdjęć do założonego wymiaru

Proszę w pętli za pomocą biblioteki Pillow oraz funkcji `Image.fromarray(...).resize(...)` zrzutować wszystkie zdjęcia (w elemencie listy pod kluczem `'X'`) z naszego zbioru do określonego rozmiaru i zapisać je do nowej listy *all_data_processed* wraz z flagą (pod kluczem `'Y'`) zmienioną na wartości 0 i 1 zgodnie ze słownikiem *label_dict* poniżej.
<br>**Jeśli sieć neuronowa nie będzie się uczyć, wskazując na brak pamięci, proszę zmienić IMG_SIZE na 32**

In [ ]:
IMG_SIZE = 64
RESAMPLING = Image.LANCZOS  # filtr próbkowania: LANCZOS = najwyższa jakość przy zmniejszaniu obrazu

number_of_examples = len(all_data)
let_know = int(number_of_examples / 10)
all_data_processed = []
label_dict = {'cat': 1, 'not-cat': 0}

...
# (przejdź pętlą po all_data, każdy obrazek zrzutuj do (IMG_SIZE, IMG_SIZE) używając RESAMPLING,
#  zamień klasę przez label_dict i dopisz słownik {'X': ..., 'Y': ...} do all_data_processed)

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

Każdy element `all_data` to słownik `{'X': obraz_jako_ndarray, 'Y': 'cat' lub 'not-cat'}`.

Schemat jednej iteracji:
1. `img = Image.fromarray(example['X']).resize((IMG_SIZE, IMG_SIZE), RESAMPLING)`
2. zamień klasę: `label_dict[example['Y']]`
3. dopisz `{'X': np.array(img), 'Y': ...}` do `all_data_processed`

Pamiętaj o postępie — wewnątrz pętli można drukować informację co `let_know` przykładów.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
for idx, example in enumerate(all_data):
    if (idx + 1) % let_know == 0:
        print(f'processing {idx + 1}')

    resized_down = Image.fromarray(example['X']).resize((IMG_SIZE, IMG_SIZE), RESAMPLING)
    all_data_processed.append({'X': np.array(resized_down), 'Y': label_dict[example['Y']]})

##### Zobaczmy kilka zdjęć naszego zbioru

In [ ]:
classNames = {value: key for key, value in label_dict.items()}
fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 12))
axes = axes.flatten()
number_of_examples = len(all_data_processed)
for axis in axes:
    idx = random.randint(0, number_of_examples - 1)
    example = all_data_processed[idx]
    axis.axis('off')
    axis.set_title(f"{classNames[example['Y']]}")
    axis.imshow(example['X'])

##### Proszę o losowe przemieszanie listy danych
Po wczytaniu danych warto je losowo przemieszać — często zbiory są uporządkowane wg klas (np. najpierw same koty, potem same nie-koty), co psułoby podział na zbiór treningowy/testowy.

In [ ]:
type(all_data_processed)

In [ ]:
random.seed(42)
...

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

`random.shuffle(...)` przemiesza listę **w miejscu** (in-place) — nie trzeba przypisywać wyniku.
Wcześniejszy `random.seed(42)` gwarantuje powtarzalność tasowania (każdorazowo ten sam wynik).

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
random.seed(42)
random.shuffle(all_data_processed)

##### Sprawdźmy wymiary naszych X'ów i Y'ków

In [ ]:
X = np.array([example['X'] for example in all_data_processed])
Y = np.array([example['Y'] for example in all_data_processed])
print(f"{bcolors.BOLD}Rozmiar cech (X): {X.shape}, rozmiar flagi/indykatora klasy (Y): {Y.shape}{bcolors.ENDC}")

##### Proszę kontynuować kod i stworzyć podzbiory
Po tych wszystkich analizach możemy w końcu podzielić nasze dane na zbiór treningowy i testowy.
##### *Jeśli sieć neuronowa nie będzie się uczyć, wskazując na brak pamięci, proszę znacząco zmniejszyć split_ratio, np. do 0.2*

> **Ważna zmiana w porównaniu do wersji TensorFlow/Keras:**
>
> W PyTorch używamy `nn.CrossEntropyLoss`, która **wewnętrznie** stosuje softmax i oczekuje **etykiet jako liczb całkowitych** (np. 0, 1), a **nie** wektorów one-hot.
> Dlatego **nie** wykonujemy kodowania one-hot na Y — zostawiamy Y jako wektor jednoelementowych etykiet (np. `[0, 1, 1, 0, ...]`).

In [ ]:
split_ratio = 0.6
split_idx = int(len(Y) * split_ratio)

# W PyTorch NIE stosujemy OneHotEncoder — CrossEntropyLoss oczekuje etykiet integer
# Proszę zadeklarować poniższe zmienne korzystając z macierzy X i Y (podpowiedź: slice)
X_train = None
X_test = None
Y_train = None
Y_test = None

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

`X` ma kształt `(N, 64, 64, 3)` — slice po pierwszej osi:
- `X_train = X[:split_idx]` (pierwsze `split_idx` próbek)
- `X_test  = X[split_idx:]` (pozostałe)

Analogicznie dla `Y`. W PyTorch `Y` zostaje jako wektor int-ów `(N,)` — `CrossEntropyLoss` sam to obsłuży.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
X_train = X[:split_idx]
X_test = X[split_idx:]
Y_train = Y[:split_idx]
Y_test = Y[split_idx:]

##### Przypominam tylko, że jest gotowa funkcja do podziału na zbiór treningowy i testowy - w przyszłości polecam stosować (na razie proszę podzielić tę próbkę ręcznie)<br>
from sklearn.model_selection import train_test_split

In [ ]:
from sklearn.model_selection import train_test_split

Teraz "rozplączemy" macierze zdjęć 64x64x3 na długie wektory tworząc *X_train_flat* oraz *X_test_flat* <br><br>
**Różnica w stosunku do wersji Keras:** W PyTorch z `nn.CrossEntropyLoss` warstwa wyjściowa zwraca surowe logity (bez softmax), a loss function sama stosuje softmax + negative log-likelihood. Etykiety Y to **integer** (0 lub 1), a nie wektory one-hot.

| Klasa | Keras (one-hot) | PyTorch (integer label) |
| --- | --- | --- |
| Kot | [0, 1] | 1 |
| Nie kot | [1, 0] | 0 |

In [ ]:
X_train_flat = X_train.reshape(X_train.shape[0], -1) / 255.
X_test_flat = X_test.reshape(X_test.shape[0], -1) / 255.

print("X_train_flat shape: " + str(X_train_flat.shape))
print("X_test_flat shape: " + str(X_test_flat.shape))
print("Y_train shape: " + str(Y_train.shape))
print("Y_test shape: " + str(Y_test.shape))

Zostawimy również zdjęcia w postaci macierzy trzywymiarowej (3D) aby wykorzystać w modelu sieci neuronowej przystosowanej do otrzymywania pełnych, a nie zrzutowanych do jednego wektora, obrazków

In [ ]:
print("X_train shape: " + str(X_train.shape))
print("X_test shape: " + str(X_test.shape))

##### Konwersja danych do tensorów PyTorch

W PyTorch dane muszą być tensorami (`torch.Tensor`). Konwertujemy numpy arrays na tensory i przenosimy je na wybrane urządzenie (MPS/CUDA/CPU).

In [ ]:
# Tensory dla sieci pełnych połączeń (FFNN i Dropout) — spłaszczone obrazki
X_train_flat_t = torch.tensor(X_train_flat, dtype=torch.float32)
X_test_flat_t = torch.tensor(X_test_flat, dtype=torch.float32)

# Etykiety jako long (int64) — wymagane przez CrossEntropyLoss
Y_train_t = torch.tensor(Y_train, dtype=torch.long)
Y_test_t = torch.tensor(Y_test, dtype=torch.long)

print(f"X_train_flat_t: {X_train_flat_t.shape}, dtype={X_train_flat_t.dtype}")
print(f"Y_train_t: {Y_train_t.shape}, dtype={Y_train_t.dtype}")

##### Funkcja treningowa

W PyTorch (w odróżnieniu od Keras) nie ma wbudowanego `model.fit()`. Poniższa funkcja `train_model` pełni tę samą rolę — trenuje model i zwraca historię metryk (accuracy, loss) w formacie identycznym jak `history.history` w Keras.

In [ ]:
def train_model(model, X_train, Y_train, X_val, Y_val,
                epochs=50, batch_size=32, lr=1e-3, device='cpu'):
    """
    Trenuje model PyTorch i zwraca historię metryk.

    Zwraca dict z kluczami: 'accuracy', 'val_accuracy', 'loss', 'val_loss'
    (identycznie jak Keras history.history).
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # DataLoader do minibatch-owania
    train_dataset = TensorDataset(X_train, Y_train)
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

    # Dane walidacyjne — przenosimy raz na urządzenie
    X_val_dev = X_val.to(device)
    Y_val_dev = Y_val.to(device)

    history = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': []}

    for epoch in range(epochs):
        # ---------- TRENING ----------
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for X_batch, Y_batch in train_loader:
            X_batch, Y_batch = X_batch.to(device), Y_batch.to(device)

            optimizer.zero_grad()
            outputs = model(X_batch)          # surowe logity
            loss = criterion(outputs, Y_batch)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * X_batch.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == Y_batch).sum().item()
            total += Y_batch.size(0)

        train_loss = running_loss / total
        train_acc = correct / total

        # ---------- WALIDACJA ----------
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val_dev)
            val_loss = criterion(val_outputs, Y_val_dev).item()
            _, val_predicted = torch.max(val_outputs, 1)
            val_acc = (val_predicted == Y_val_dev).sum().item() / Y_val_dev.size(0)

        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        print(f"Epoch {epoch+1:3d}/{epochs} — "
              f"loss: {train_loss:.4f} — accuracy: {train_acc:.4f} — "
              f"val_loss: {val_loss:.4f} — val_accuracy: {val_acc:.4f}")

    return history

## Feed Forward Neural Network

Poniżej stworzymy architekturę pierwszej sieci neuronowej — jest to sieć pełnych połączeń (Feed Forward Neural Network). Wykorzystamy do tego PyTorch i moduł `nn.Sequential` aby zbudować kolejne warstwy sieci.<br><br>
Sieć jest w pewnym sensie kilkoma ułożonymi w warstwy regresjami logistycznymi (tylko często używamy innej funkcji aktywacji niż sigmoida) — czyli np. wyjścia pierwszej regresji logistycznej (pierwszej warstwy) są wejściami dla drugiej regresji logistycznej (drugiej warstwy).<br><br>
Deep Learningiem (uczeniem głębokim) nazywamy struktury w których występuje właśnie taka hierarchia, łańcuch warstw. Główną ideą za tym stojącą jest przypuszczenie, że taki model wyuczy się pewnych meta-cech w warstwach pośrednich/ukrytych. Na przykład, być może sieć uczona na danych dotyczących mieszkań (regresja liniowa) stworzyłaby metacechę oznaczającą jak przestrzenne jest mieszkanie — być może jakaś kombinacja cech wejściowych "wytworzy" taką meta-cechę. Musimy jednak pamiętać, że raczej nie jesteśmy w stanie "odczytać" tych metacech ani ręcznie ich ustawić.<br><br>
**Proszę uzupełnić poniższe linijki (wartości `None` i `"please_fill"`).** Można korzystać z dokumentacji. Również poniżej (po podpowiedzi i rozwiązaniu) sprawdzamy architekturę sieci.

> **Uwaga PyTorch:** W `nn.Sequential` nie podajemy `input_dim` — warstwa `nn.Linear(in_features, out_features)` sama definiuje rozmiar wejścia jako pierwszy argument. Nie dodajemy warstwy `Softmax` na końcu, ponieważ `nn.CrossEntropyLoss` zawiera ją wewnętrznie.

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
try:
    model = nn.Sequential(
        # Pierwsza warstwa: wejście = IMG_SIZE*IMG_SIZE*3, wyjście = 1024 neuronów, aktywacja ReLU
        nn.Linear(None, None),
        nn.ReLU(),

        # Druga warstwa: 512 neuronów, aktywacja ReLU
        nn.Linear(None, None),
        nn.ReLU(),

        # Warstwa wyjściowa: liczba neuronów = liczba klas (2)
        # BEZ softmax — CrossEntropyLoss stosuje go wewnętrznie
        nn.Linear(None, None),
    )
    # Szybki test: sprawdź czy model przyjmuje dane
    _test = model(torch.randn(1, IMG_SIZE * IMG_SIZE * 3))
    print(f"Model OK — wyjście: {_test.shape}")

except Exception:
    print(f'{bcolors.BOLD}{bcolors.FAIL}Proszę poprawnie uzupełnić powyższe miejsca gdzie występuje None lub "please_fill"{bcolors.ENDC}')

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

`nn.Linear(in_features, out_features)` to odpowiednik `Dense(out_features, input_dim=in_features)` z Keras.

Dla obrazka 64×64×3 rozwiniętego do wektora: `in_features = IMG_SIZE * IMG_SIZE * 3 = 12288`.

Schemat:
```python
nn.Linear(12288, 1024),  # 1024 neuronów
nn.ReLU(),
nn.Linear(1024, 512),    # 512 neuronów
nn.ReLU(),
nn.Linear(512, 2),       # 2 klasy (bez softmax!)
```

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
np.random.seed(42)
torch.manual_seed(42)
model = nn.Sequential(
    nn.Linear(IMG_SIZE * IMG_SIZE * 3, 1024),
    nn.ReLU(),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Linear(512, 2),
)

##### Sprawdźmy jak wygląda nasza sieć

In [ ]:
torchinfo_summary(model, input_size=(1, IMG_SIZE * IMG_SIZE * 3))

###### <span style="color: #4a7090;">📊 Spodziewany wynik</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

```
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
├─Linear: 1-1                            [1, 1024]                 12,583,936
├─ReLU: 1-2                              [1, 1024]                 --
├─Linear: 1-3                            [1, 512]                  524,800
├─ReLU: 1-4                              [1, 512]                  --
├─Linear: 1-5                            [1, 2]                    1,026
==========================================================================================
Total params: 13,109,762
```

> **Liczba parametrów: 13 109 762 (~13.1 M).**
> To bardzo dużo wag — sieć ma sporą pojemność i potrafi „nauczyć się na pamięć" zbioru treningowego.
> Spodziewany efekt: silny **overfit** — train accuracy szybko rośnie, val accuracy wyraźnie zostaje w tyle (gap ≈ 0.15–0.20).

###### 

In [ ]:
print(model)

##### Trenowanie modelu

Powyżej zadeklarowaliśmy architekturę naszej sieci (warstwy, neurony, funkcje aktywacji itd.), jednak aby model mógł się uczyć musimy podać jeszcze szczegóły dotyczące iteracyjnego poruszania w stronę gradientu. W poniższej komórce proszę uzupełnić wywołanie funkcji `train_model` zgodnie z podaną instrukcją.

> W PyTorch nie ma `model.compile()` + `model.fit()` — zamiast tego używamy napisanej wyżej funkcji `train_model()`.

In [ ]:
try:
    # Trening: model, X treningowe (flat tensor), Y treningowe (tensor),
    # X walidacyjne (flat tensor), Y walidacyjne (tensor),
    # liczba epok = 50, batch_size = 32, device = device
    history_model = train_model(
        None, None, None,
        None, None,
        epochs=None, batch_size=None, device=device,
    )
except Exception:
    print(f'{bcolors.BOLD}{bcolors.FAIL}Proszę poprawnie uzupełnić powyższe miejsca gdzie występuje None{bcolors.ENDC}')

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

```python
history_model = train_model(
    model, X_train_flat_t, Y_train_t,
    X_test_flat_t, Y_test_t,
    epochs=50, batch_size=32, device=device,
)
```

Funkcja `train_model` sama przenosi dane na urządzenie (MPS/CPU), tworzy DataLoader i prowadzi pętlę treningową.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
history_model = train_model(
    model, X_train_flat_t, Y_train_t,
    X_test_flat_t, Y_test_t,
    epochs=50, batch_size=32, device=device,
)

##### **Pod koniec uczenia dokładność (accuracy) dla zbioru treningowego powinna być powyżej 80% zaś dla zbioru testowego (val) około 65%.**

Poniżej wyrysujemy wykres dokładności (accuracy) zbioru treningowego i testowego w zależności od epok/czasu uczenia. Proszę zauważyć, że od pewnego momentu to raczej dokładność na zbiorze treningowym rośnie, a na testowym stoi w miejscu - być może następuje overfitting.

In [ ]:
plt.plot(history_model['accuracy'])
plt.plot(history_model['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.ylim(ymin=0, ymax=1)
plt.show()

Poniżej wykres funkcji kosztu zbioru treningowego i testowego - proszę pamiętać, że uczenie modelu odbywa się poprzez iteracyjne zmniejszanie funkcji kosztu zbioru treningowego, ale model nawet nie powinien znać tej wartości dla zbioru testowego. Proszę zwrócić uwagę jak na początku funkcja kosztu zbioru testowego maleje, ale po chwili zaczyna rosnąć - jest to oczywiście niepożądany skutek i najprawdopodobniej miejsce, od którego zaczynamy "overfittować" model.

In [ ]:
plt.plot(history_model['loss'])
plt.plot(history_model['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'test'], loc='upper left')
plt.show()

### Dropout
Dropout jest techniką regularyzacji, która podczas uczenia losowo "ucina/zakrywa" połączenia w sieci neuronowej przez co sygnał nie będzie propagowany. Intuicja za tym stojąca jest następująca — to tak, jakby model za każdym razem nie miał dostępu do wszystkich cech: czy wprawny analityk banku będzie w stanie oszacować, czy kredyt powinien być przyznany, gdyby zabrakło mu jednej danej z formularza?

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
try:
    model_reg = nn.Sequential(
        # Input dropout — prawdopodobieństwo ucięcia 0.2 (20%)
        nn.Dropout(p=None),

        nn.Linear(IMG_SIZE * IMG_SIZE * 3, 1024),
        nn.ReLU(),

        # Dropout 0.2 (20%)
        nn.Dropout(p=None),

        nn.Linear(1024, 512),
        nn.ReLU(),

        # Dropout 0.2 (20%)
        nn.Dropout(p=None),

        # Warstwa wyjściowa (bez softmax)
        nn.Linear(512, 2),
    )
    # Szybki test
    _test = model_reg(torch.randn(1, IMG_SIZE * IMG_SIZE * 3))
    print(f"Model OK — wyjście: {_test.shape}")

except Exception:
    print(f'{bcolors.BOLD}{bcolors.FAIL}Proszę poprawnie uzupełnić powyższe miejsca gdzie występuje None{bcolors.ENDC}')

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

`nn.Dropout(p=0.2)` losowo "wyłącza" 20% połączeń podczas uczenia.
W PyTorch `p` to prawdopodobieństwo zerowania (to samo co `rate` w Keras).
Wartość `p` to liczba między 0.0 a 1.0.

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
np.random.seed(42)
torch.manual_seed(42)
model_reg = nn.Sequential(
    nn.Dropout(p=0.2),
    nn.Linear(IMG_SIZE * IMG_SIZE * 3, 1024),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(1024, 512),
    nn.ReLU(),
    nn.Dropout(p=0.2),
    nn.Linear(512, 2),
)

##### Sprawdźmy jak wygląda nasza sieć

In [ ]:
torchinfo_summary(model_reg, input_size=(1, IMG_SIZE * IMG_SIZE * 3))

###### <span style="color: #4a7090;">📊 Spodziewany wynik</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

```
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
├─Dropout: 1-1                           [1, 12288]                --
├─Linear: 1-2                            [1, 1024]                 12,583,936
├─ReLU: 1-3                              [1, 1024]                 --
├─Dropout: 1-4                           [1, 1024]                 --
├─Linear: 1-5                            [1, 512]                  524,800
├─ReLU: 1-6                              [1, 512]                  --
├─Dropout: 1-7                           [1, 512]                  --
├─Linear: 1-8                            [1, 2]                    1,026
==========================================================================================
Total params: 13,109,762
```

> **Liczba parametrów: 13 109 762 (~13.1 M) — identyczna jak FFNN.**
> Warstwy `Dropout` **nie mają wag** (same w sobie nie dodają parametrów), a same warstwy `Linear` są dokładnie takie same jak w modelu bazowym.
> Mimo identycznej pojemności **overfit dramatycznie spada** (gap z ≈ 0.20 do ≈ 0.07), a val accuracy jest porównywalny lub nieznacznie lepszy.
> Lekcja: **dropout to czysta regularyzacja — nie zwiększa pojemności, ale uczy sieć generalizować zamiast zapamiętywać.**

###### 

In [ ]:
print(model_reg)

##### Trenowanie modelu z regularyzacją

In [ ]:
# rozwiązanie
# Apples-to-apples z FFNN: ten sam optimizer (Adam) i batch_size, jedyną różnicą są warstwy Dropout
history_model_reg = train_model(
    model_reg, X_train_flat_t, Y_train_t,
    X_test_flat_t, Y_test_t,
    epochs=50, batch_size=32, device=device,
)

**Pod koniec uczenia dokładność (accuracy) dla zbioru treningowego powinna być blisko 80% zaś dla zbioru testowego (val) powyżej 70%.**
<br><br>
Poniżej wyrysujemy wykres dokładności (accuracy) zbioru treningowego i testowego w zależności od epok/czasu uczenia. Proszę zauważyć, że rośnie dokładność na obu zbiorach i różnica w dokładności pomiędzy nimi w czasie nie jest aż tak duża jak w przypadku modelu bez dropout - to właśnie chcieliśmy osiągnąć.

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 7))

axes[0].plot(history_model_reg['accuracy'])
axes[0].plot(history_model_reg['val_accuracy'])
axes[0].set_title('regularized model accuracy')
axes[0].set_ylabel('accuracy')
axes[0].set_xlabel('epoch')
axes[0].legend(['train', 'test'], loc='upper left')
axes[0].set_ylim(ymin=0, ymax=1)

axes[1].plot(history_model['accuracy'])
axes[1].plot(history_model['val_accuracy'])
axes[1].set_title('simple model accuracy')
axes[1].set_ylabel('accuracy')
axes[1].set_xlabel('epoch')
axes[1].legend(['train', 'test'], loc='upper left')
axes[1].set_ylim(ymin=0, ymax=1)
plt.show()

Poniżej porównanie funkcji kosztu dla prostego modelu NN i z regularyzacją *dropout*

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(20, 7))

y_max = max(
    max(history_model_reg['loss']), max(history_model_reg['val_loss']),
    max(history_model['loss']), max(history_model['val_loss']),
)

axes[0].plot(history_model_reg['loss'])
axes[0].plot(history_model_reg['val_loss'])
axes[0].set_title('regularized model loss')
axes[0].set_ylabel('loss')
axes[0].set_xlabel('epoch')
axes[0].legend(['train', 'test'], loc='upper left')
axes[0].set_ylim(ymin=0, ymax=y_max)

axes[1].plot(history_model['loss'])
axes[1].plot(history_model['val_loss'])
axes[1].set_title('simple model loss')
axes[1].set_ylabel('loss')
axes[1].set_xlabel('epoch')
axes[1].legend(['train', 'test'], loc='upper left')
axes[1].set_ylim(ymin=0, ymax=y_max)

plt.show()

## Konwolucyjna sieć neuronowa (CNN)

Ta architektura sieci jest jedną z najczęściej doradzanych w przypadku pracy nad obrazkami, gdyż może ona wziąć pod uwagę pewne "lokalności" znajdujące się na obrazku. Główną ideą w konwolucyjnych sieciach są tzw. filtry, które przesuwają się po obrazku jak okno przesuwne (sliding window) i operują na lokalnych fragmentach. Podczas uczenia te filtry mogą wyłapać pewne małe kształty, które później sieć może złożyć w bardziej zaawansowane struktury.<br>
Ponieważ chcemy pracować na pełnych obrazkach (macierz 3D) to weźmiemy znormalizowane dane (64x64) X_train i X_test zamiast X_train_flat i X_test_flat.

> **Uwaga PyTorch:** Konwencja wymiarów w PyTorch to `(N, C, H, W)` — kanały kolorów **przed** wymiarami przestrzennymi. W numpy/Keras jest `(N, H, W, C)`. Musimy więc transponować dane.

In [ ]:
print("X_train shape: " + str(X_train.shape))
print("X_test shape: " + str(X_test.shape))
print("Y_train shape: " + str(Y_train.shape))
print("Y_test shape: " + str(Y_test.shape))

In [ ]:
# Normalizacja i konwersja do tensorów PyTorch
# WAŻNE: PyTorch CNN oczekuje (N, C, H, W), a numpy ma (N, H, W, C)
# Dlatego transponujemy osie: (0, 3, 1, 2) zamienia (N, H, W, C) → (N, C, H, W)
X_train_cnn = torch.tensor(X_train / 255., dtype=torch.float32).permute(0, 3, 1, 2)
X_test_cnn = torch.tensor(X_test / 255., dtype=torch.float32).permute(0, 3, 1, 2)

print(f"X_train_cnn shape: {X_train_cnn.shape}  (N, C, H, W)")
print(f"X_test_cnn shape:  {X_test_cnn.shape}")

In [ ]:
np.random.seed(42)
torch.manual_seed(42)
try:
    model_cnn = nn.Sequential(
        # Liczba filtrów = 32, rozmiar filtra 3×3, padding "same" (=1 dla 3×3), aktywacja ReLU
        nn.Conv2d(in_channels=None, out_channels=None, kernel_size=None, padding=None),
        nn.ReLU(),
        # MaxPooling z rozmiarem 2×2
        nn.MaxPool2d(kernel_size=None),
        nn.Dropout(p=0.3),

        # Liczba filtrów = 64, reszta jak wyżej
        nn.Conv2d(in_channels=None, out_channels=None, kernel_size=None, padding=None),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=None),
        nn.Dropout(p=0.3),

        # Liczba filtrów = 64, reszta jak wyżej
        nn.Conv2d(in_channels=None, out_channels=None, kernel_size=None, padding=None),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=None),
        nn.Dropout(p=0.3),

        # Rzutujemy wszystko na jeden długi wektor
        nn.Flatten(),
        nn.Linear(None, 128),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(128, 2),
    )
    # Szybki test
    _test = model_cnn(torch.randn(1, 3, IMG_SIZE, IMG_SIZE))
    print(f"Model OK — wyjście: {_test.shape}")

except Exception as e:
    print(f'{bcolors.BOLD}{bcolors.FAIL}Proszę poprawnie uzupełnić powyższe miejsca gdzie występuje None: {e}{bcolors.ENDC}')

###### <span style="color: #c17f24;">💡 Podpowiedź</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

**nn.Conv2d** — `in_channels` to liczba kanałów wejściowych (3 dla RGB, potem tyle ile filtrów poprzedniej warstwy), `out_channels` to liczba filtrów. `kernel_size=3` i `padding=1` daje `padding='same'` (zachowuje wymiary przestrzenne).

**nn.MaxPool2d** — `kernel_size=2` zmniejsza wymiary przestrzenne dwukrotnie.

**nn.Flatten** po trzech blokach Conv+Pool: 64×64 → 32×32 → 16×16 → 8×8, więc `nn.Flatten()` daje 64 × 8 × 8 = **4096**.

```python
nn.Conv2d(3, 32, kernel_size=3, padding=1),   # (N, 3, 64, 64) → (N, 32, 64, 64)
nn.Conv2d(32, 64, kernel_size=3, padding=1),   # (N, 32, 32, 32) → (N, 64, 32, 32)
nn.Conv2d(64, 64, kernel_size=3, padding=1),   # (N, 64, 16, 16) → (N, 64, 16, 16)
nn.Linear(64 * 8 * 8, 128),                    # 4096 → 128
```

###### <span style="color: #5a8a6a;">✅ Rozwiązanie</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

In [ ]:
# rozwiązanie
np.random.seed(42)
torch.manual_seed(42)
model_cnn = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Dropout(p=0.3),

    nn.Conv2d(32, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Dropout(p=0.3),

    nn.Conv2d(64, 64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2),
    nn.Dropout(p=0.3),

    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 128),
    nn.ReLU(),
    nn.Dropout(p=0.3),
    nn.Linear(128, 2),
)

##### Sprawdźmy jak wygląda nasza sieć

In [ ]:
torchinfo_summary(model_cnn, input_size=(1, 3, IMG_SIZE, IMG_SIZE))

###### <span style="color: #4a7090;">📊 Spodziewany wynik</span> <span style="color: #999; font-weight: normal; font-size: 0.85em;">(kliknij aby rozwinąć)</span>

```
==========================================================================================
Layer (type:depth-idx)                   Output Shape              Param #
==========================================================================================
├─Conv2d: 1-1                            [1, 32, 64, 64]           896
├─ReLU: 1-2                              [1, 32, 64, 64]           --
├─MaxPool2d: 1-3                         [1, 32, 32, 32]           --
├─Dropout: 1-4                           [1, 32, 32, 32]           --
├─Conv2d: 1-5                            [1, 64, 32, 32]           18,496
├─ReLU: 1-6                              [1, 64, 32, 32]           --
├─MaxPool2d: 1-7                         [1, 64, 16, 16]           --
├─Dropout: 1-8                           [1, 64, 16, 16]           --
├─Conv2d: 1-9                            [1, 64, 16, 16]           36,928
├─ReLU: 1-10                             [1, 64, 16, 16]           --
├─MaxPool2d: 1-11                        [1, 64, 8, 8]             --
├─Dropout: 1-12                          [1, 64, 8, 8]             --
├─Flatten: 1-13                          [1, 4096]                 --
├─Linear: 1-14                           [1, 128]                  524,416
├─ReLU: 1-15                             [1, 128]                  --
├─Dropout: 1-16                          [1, 128]                  --
├─Linear: 1-17                           [1, 2]                    258
==========================================================================================
Total params: 580,994
```

> **Liczba parametrów: 580 994 (~581 K) — ponad 22× mniej niż FFNN/Dropout.**
> A jednak CNN osiąga val accuracy ≈ 0.89–0.90, podczas gdy FFNN/Dropout zatrzymują się na ≈ 0.72.
> Powód: konwolucje mają **właściwy *inductive bias* dla obrazów** — wykrywają lokalne wzorce niezależnie od pozycji.

###### 

In [ ]:
print(model_cnn)

##### Trenowanie modelu CNN

In [ ]:
history_model_cnn = train_model(
    model_cnn, X_train_cnn, Y_train_t,
    X_test_cnn, Y_test_t,
    epochs=50, batch_size=64, device=device,
)

**Pod koniec uczenia dokładność (accuracy) dla zbioru treningowego powinna być powyżej 90% zaś dla zbioru testowego (val) około 90%.**
<br><br>
Poniżej wyrysujemy wykres dokładności (accuracy) zbioru treningowego i testowego w zależności od epok/czasu uczenia oraz funkcji kosztu.

In [ ]:
plt.plot(history_model_cnn['accuracy'])
plt.plot(history_model_cnn['val_accuracy'])
plt.title('model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.ylim(ymin=0, ymax=1)
plt.show()

In [ ]:
plt.plot(history_model_cnn['loss'])
plt.plot(history_model_cnn['val_loss'])
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train', 'val'], loc='upper left')
plt.show()

## Porównanie wszystkich trzech modeli

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 7))

histories = [
    (history_model, 'FFNN (simple)'),
    (history_model_reg, 'FFNN + Dropout'),
    (history_model_cnn, 'CNN'),
]

for ax, (hist, title) in zip(axes, histories):
    ax.plot(hist['accuracy'], label='train')
    ax.plot(hist['val_accuracy'], label='val')
    ax.set_title(f'{title} — accuracy')
    ax.set_ylabel('accuracy')
    ax.set_xlabel('epoch')
    ax.legend(loc='upper left')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(24, 7))

y_max_all = max(
    max(h['loss'] + h['val_loss']) for h, _ in histories
)

for ax, (hist, title) in zip(axes, histories):
    ax.plot(hist['loss'], label='train')
    ax.plot(hist['val_loss'], label='val')
    ax.set_title(f'{title} — loss')
    ax.set_ylabel('loss')
    ax.set_xlabel('epoch')
    ax.legend(loc='upper left')
    ax.set_ylim(0, y_max_all)

plt.tight_layout()
plt.show()

## Sprawdź swoje zdjęcie

W poniższych komórkach można podać swoje zdjęcie (z dysku lub internetu) aby na żywo dokonać predykcji.
<br><br>
Funkcje `predict_image_pytorch` oraz `predict_urls_pytorch` poniżej obsługują PyTorch modele (FFNN, Dropout, CNN).

In [ ]:
def predict_image_pytorch(models, image, classNames, img_size=64):
    """
    Predykcja dla jednego obrazka PIL na liście modeli PyTorch.
    Zwraca dict {nazwa_modelu: nazwa_klasy}.
    """
    results = {}
    for name, m in models.items():
        m.eval()
        resized = image.resize((img_size, img_size), Image.LANCZOS)
        arr = np.array(resized) / 255.0

        # Sprawdzamy czy model oczekuje danych 2D (flat) czy 4D (CNN)
        first_layer = list(m.children())[0]
        is_cnn = isinstance(first_layer, (nn.Conv2d, nn.BatchNorm2d))
        # Jeśli pierwszy layer to Dropout, sprawdź następny
        if isinstance(first_layer, nn.Dropout):
            children = list(m.children())
            is_cnn = len(children) > 1 and isinstance(children[1], (nn.Conv2d, nn.BatchNorm2d))

        if is_cnn:
            x = torch.tensor(arr, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0)
        else:
            x = torch.tensor(arr.reshape(1, -1), dtype=torch.float32)

        with torch.no_grad():
            out = m.cpu()(x)
            pred = torch.argmax(out, dim=1).item()
        results[name] = classNames[pred]
    return results


def predict_urls_pytorch(models, urls, classNames, img_size=64):
    """Predykcja dla listy URLi na wielu modelach PyTorch."""
    cols = 2 if len(urls) >= 2 else 1
    rows = int(np.ceil(len(urls) / cols))
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(18, 6 * rows))

    if len(urls) >= 2:
        axes = axes.flatten()
    else:
        axes = [axes]

    for photo_url, axis in zip(urls, axes):
        url_response = urllib.request.urlopen(photo_url)
        img_array = np.array(bytearray(url_response.read()), dtype=np.uint8)
        image = Image.open(io.BytesIO(img_array)).convert('RGB')
        axis.imshow(image)

        results = predict_image_pytorch(models, image, classNames, img_size)
        title_lines = [f"{name}: {cls}" for name, cls in results.items()]
        axis.set_title("\n".join(title_lines), fontsize=11)
        axis.axis('off')

    # Ukryj puste subploty
    for i in range(len(urls), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()


def predict_files_pytorch(models, file_paths, classNames, img_size=64):
    """Predykcja dla listy ścieżek plików na wielu modelach PyTorch."""
    cols = 2 if len(file_paths) >= 2 else 1
    rows = int(np.ceil(len(file_paths) / cols))
    fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(18, 6 * rows))

    if len(file_paths) >= 2:
        axes = axes.flatten()
    else:
        axes = [axes]

    for image_path, axis in zip(file_paths, axes):
        axis.axis('off')
        image = Image.open(image_path).convert('RGB')
        axis.imshow(image)

        results = predict_image_pytorch(models, image, classNames, img_size)
        title_lines = [f"{name}: {cls}" for name, cls in results.items()]
        axis.set_title("\n".join(title_lines), fontsize=11)

    for i in range(len(file_paths), len(axes)):
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:
# Słownik modeli do predykcji
all_models = {
    "FFNN": model,
    "Dropout": model_reg,
    "CNN": model_cnn,
}

In [ ]:
# Predykcja z plików (tkinter file dialog)
my_button = SelectFilesButton()
my_button  # wyświetla przycisk wyboru plików

In [ ]:
predict_files_pytorch(all_models, my_button.files, classNames)

In [ ]:
# Przykładowe URLe do testowania — odkomentuj które chcesz.
# TP = True Positive, TN = True Negative, FP = False Positive, FN = False Negative

urls = [
    'https://raw.githubusercontent.com/fastai/fastai/main/nbs/images/cat.jpg',                                          # TP — kot
    'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n02123597_Siamese_cat.JPEG',            # TP — kot syjamski
    # 'https://upload.wikimedia.org/wikipedia/commons/1/15/Cat_August_2010-4.jpg',                                       # TP — kot pręgowany
    # 'https://upload.wikimedia.org/wikipedia/commons/7/71/Nursing_Cat_01.jpg',                                          # TP — kotka z kociętami
    # 'https://upload.wikimedia.org/wikipedia/commons/b/b6/Felis_catus-cat_on_snow.jpg',                                 # TP — kot na śniegu
    # 'https://upload.wikimedia.org/wikipedia/commons/9/9b/Gustav_chocolate.jpg',                                        # FN — brązowy kot, model się myli
    # 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image1.jpg',      # FP — dwa psy beagle, model się myli
    # 'https://raw.githubusercontent.com/tensorflow/models/master/research/object_detection/test_images/image2.jpg',      # TN — kitesurferzy na plaży
    # 'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n01614925_bald_eagle.JPEG',            # TN — bielik
    # 'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n07734744_mushroom.JPEG',              # TN — grzyb
    # 'https://raw.githubusercontent.com/EliSchwartz/imagenet-sample-images/master/n03445777_golf_ball.JPEG',             # TN — piłka golfowa
]

predict_urls_pytorch(all_models, urls, classNames)